In [3]:
print(0)

0


In [4]:
import os
import pandas as pd

PRESCREENING_ROOT = "/home/pratyushprasad/ACL/DATA/PRESCREENING"

FOLDERS = {
    "human_derm":  ("HUMAN_DERMATOLOGY",      "LABEL", "NO"),
    "img_quality": ("IMAGE_QUALITY",           "LABEL", "BAD"),
    "micro_derm":  ("MICROSCOPIC_DERMOSCOPIC", "LABEL", "REJECT"),
    "surgery":     ("SURGERY",                 "LABEL", "REJECT"),
}

DATASETS = {
    "1. DERM_1M": [
        "derm_1m_iiyi", "derm_1m_edu", "derm_1m_note", "derm_1m_pubmed",
        "derm_1m_public", "derm_1m_reddit", "derm_1m_twitter",
        "derm_1m_validation", "derm_1m_youtube_1", "derm_1m_youtube_2",
        "derm_1m_youtube_3", "derm_1m_youtube_4",
    ],
    "2. DERM_VQA":         ["derm_vqa"],
    "3. MM_SKIN_QA":       ["mm_skin"],
    "4. PAD_UFES_20":      ["pad_ufes"],
    "5. SCIN":             ["scin"],
    "6. BIOMEDICA_PUBMED": ["biomedica"],
    "7. FITZPATRIK_17K":   ["fitzpatrick_17k"],
    "8. MUHOMMAD_DATA":    ["muhommad"],
    "9. YUHOS16_SKINCARE": ["yuhos"],
}

IMAGE_PATH_COL = "image_path"


def load_csvs(folder_path, stems):
    frames = []
    for stem in stems:
        fpath = os.path.join(folder_path, f"{stem}.csv")
        if os.path.exists(fpath):
            try:
                frames.append(pd.read_csv(fpath, dtype=str))
            except Exception as e:
                print(f"  [WARN] could not read {fpath}: {e}")
        else:
            print(f"  [WARN] missing: {fpath}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def build_merged(stems):
    """
    Load all 4 folder CSVs for this dataset and merge on image_path.
    Returns a DataFrame with columns:
      image_path, label_human, label_quality, label_micro, label_surgery
    """
    dfs = {}
    for key, (folder_name, col, _) in FOLDERS.items():
        fpath = os.path.join(PRESCREENING_ROOT, folder_name)
        df    = load_csvs(fpath, stems)
        if df.empty or IMAGE_PATH_COL not in df.columns:
            dfs[key] = pd.DataFrame(columns=[IMAGE_PATH_COL, f"label_{key}"])
            continue
        df = df[[IMAGE_PATH_COL, col]].copy()
        df[IMAGE_PATH_COL] = df[IMAGE_PATH_COL].str.strip()
        df[col]            = df[col].str.strip().str.upper()
        df = df.rename(columns={col: f"label_{key}"})
        dfs[key] = df

    merged = dfs["human_derm"]
    for key in ["img_quality", "micro_derm", "surgery"]:
        merged = pd.merge(merged, dfs[key], on=IMAGE_PATH_COL, how="outer")

    return merged


def compute_stats(merged):
    """
    For each image decide which filters reject it, return dict of counts.
    """
    total = len(merged)

    rej_human   = merged.get("label_human_derm",  pd.Series(dtype=str)).str.upper() == "NO"
    rej_quality = merged.get("label_img_quality",  pd.Series(dtype=str)).str.upper() == "BAD"
    rej_micro   = merged.get("label_micro_derm",   pd.Series(dtype=str)).str.upper() == "REJECT"
    rej_surgery = merged.get("label_surgery",       pd.Series(dtype=str)).str.upper() == "REJECT"

    # fill NaN → False
    rej_human   = rej_human.fillna(False)
    rej_quality = rej_quality.fillna(False)
    rej_micro   = rej_micro.fillna(False)
    rej_surgery = rej_surgery.fillna(False)

    any_rejected = rej_human | rej_quality | rej_micro | rej_surgery

    return {
        "total":       total,
        "human_derm":  int(rej_human.sum()),
        "img_quality": int(rej_quality.sum()),
        "micro_derm":  int(rej_micro.sum()),
        "surgery":     int(rej_surgery.sum()),
        "rejected":    int(any_rejected.sum()),       # unique images rejected by ANY filter
        "remaining":   int(total - any_rejected.sum()),
    }


def print_table(title, results, col_keys, col_headers, col_w=14, name_w=24):
    sep = "─" * (name_w + col_w * (len(col_keys) + 1) + 5)

    print("\n" + "=" * len(sep))
    print(f"  {title}")
    print("=" * len(sep))

    # header
    print(f"  {'Dataset':<{name_w}}  {'Total':>{col_w}}", end="")
    for h in col_headers:
        print(f"  {h.split(chr(10))[0]:>{col_w}}", end="")
    print()
    print(f"  {'':>{name_w}}  {'':>{col_w}}", end="")
    for h in col_headers:
        sub = h.split("\n")[1] if "\n" in h else ""
        print(f"  {sub:>{col_w}}", end="")
    print()
    print(f"  {sep}")

    totals = {"total": 0}
    for k in col_keys:
        totals[k] = 0

    for dname, row in results.items():
        totals["total"] += row["total"]
        for k in col_keys:
            totals[k] += row[k]

        print(f"  {dname:<{name_w}}  {row['total']:>{col_w},}", end="")
        for k in col_keys:
            pct = 100 * row[k] / row["total"] if row["total"] else 0
            print(f"  {row[k]:>{col_w-7},} ({pct:4.1f}%)", end="")
        print()

    print(f"  {sep}")
    print(f"  {'TOTAL':<{name_w}}  {totals['total']:>{col_w},}", end="")
    for k in col_keys:
        pct = 100 * totals[k] / totals["total"] if totals["total"] else 0
        print(f"  {totals[k]:>{col_w-7},} ({pct:4.1f}%)", end="")
    print()
    print("=" * len(sep))


def main():
    results = {}

    print("\nMerging image_path across all 4 filters per dataset...")
    for dname, stems in DATASETS.items():
        print(f"  {dname} ...", flush=True)
        merged       = build_merged(stems)
        results[dname] = compute_stats(merged)

    # ── Table 1: main prescreening summary ────────────────────────────────────
    print_table(
        title       = "PRESCREENING STATS",
        results     = results,
        col_keys    = ["human_derm", "img_quality", "micro_derm", "surgery", "remaining"],
        col_headers = [
            "Human Derm\n(NO)",
            "Img Quality\n(BAD)",
            "Micro/Derm\n(REJECT)",
            "Surgery\n(REJECT)",
            "Remaining",
        ],
    )

    # ── Table 2: rejected images breakdown ────────────────────────────────────
    print_table(
        title       = "REJECTED IMAGES  (unique per image_path, any filter)",
        results     = results,
        col_keys    = ["human_derm", "img_quality", "micro_derm", "surgery", "rejected"],
        col_headers = [
            "Human Derm\n(NO)",
            "Img Quality\n(BAD)",
            "Micro/Derm\n(REJECT)",
            "Surgery\n(REJECT)",
            "Total Rejected\n(unique)",
        ],
    )

    print()


if __name__ == "__main__":
    main()


Merging image_path across all 4 filters per dataset...
  1. DERM_1M ...
  2. DERM_VQA ...
  3. MM_SKIN_QA ...
  4. PAD_UFES_20 ...
  5. SCIN ...
  6. BIOMEDICA_PUBMED ...
  7. FITZPATRIK_17K ...
  8. MUHOMMAD_DATA ...
  9. YUHOS16_SKINCARE ...

  PRESCREENING STATS
  Dataset                            Total      Human Derm     Img Quality      Micro/Derm         Surgery       Remaining
                                                      (NO)           (BAD)        (REJECT)        (REJECT)                
  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1. DERM_1M                       423,016   22,229 ( 5.3%)   18,143 ( 4.3%)    7,788 ( 1.8%)    8,532 ( 2.0%)  384,250 (90.8%)
  2. DERM_VQA                        2,945       10 ( 0.3%)       21 ( 0.7%)        0 ( 0.0%)       12 ( 0.4%)    2,908 (98.7%)
  3. MM_SKIN_QA                     11,002      513 ( 4.7%)       44 ( 0.4%)      950 ( 8.6%)      164 ( 1.5%)    9